In [34]:
import os 
import re 
from datetime import datetime



In [35]:
def parse_teresort_output(file_path):
    metrics = {}
    job_start_time = None
    job_end_time = None
    yarn_start_time = None

    # Pattern to capture key=value lines
    kv_pattern = re.compile(r"^\s+([^=]+)=([0-9]+)")

    # Pattern to capture start and end timestamps
    start_pattern = re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d+).*terasort\.TeraSort: starting")
    end_pattern = re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d+).*terasort\.TeraSort: done")

    yarn_start_pattern = re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d+).*mapreduce\.Job: Running job")


    with open(file_path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        # Match metrics
        kv_match = kv_pattern.match(line)
        if kv_match:
            key, value = kv_match.groups()
            metrics[key.strip()] = int(value.strip())

        # Match start time
        if not job_start_time:
            start_match = start_pattern.match(line)
            if start_match:
                job_start_time = datetime.strptime(start_match.group(1), "%Y-%m-%d %H:%M:%S,%f")

        # Match YARN start time
        if not yarn_start_time:
            yarn_start_match = yarn_start_pattern.match(line)
            if yarn_start_match:
                yarn_start_time = datetime.strptime(yarn_start_match.group(1), "%Y-%m-%d %H:%M:%S,%f")

        # Match end time
        end_match = end_pattern.match(line)
        if end_match:
            job_end_time = datetime.strptime(end_match.group(1), "%Y-%m-%d %H:%M:%S,%f")

   
    if job_start_time and job_end_time:
        duration_seconds = (job_end_time - job_start_time).total_seconds()
        metrics['WallClockTimeSeconds'] = duration_seconds
    if yarn_start_time and job_end_time:
        duration_seconds = (job_end_time - yarn_start_time).total_seconds()
        metrics['PureJobDurationSeconds'] = duration_seconds
    else:
        metrics['WallClockTimeSeconds'] = None

    return metrics

In [36]:

hadoop_log_dir = './hadooplogs' 


log_files = [os.path.join(hadoop_log_dir, f) for f in os.listdir(hadoop_log_dir) if os.path.isfile(os.path.join(hadoop_log_dir, f))]


print(log_files[0])

terasort_metrics = parse_teresort_output(log_files[0])
print(terasort_metrics)

./hadooplogs/terasort_output_6_node.txt
{'FILE: Number of bytes read': 581583903, 'FILE: Number of bytes written': 874261037, 'FILE: Number of read operations': 0, 'FILE: Number of large read operations': 0, 'FILE: Number of write operations': 0, 'HDFS: Number of bytes read': 1000000832, 'HDFS: Number of bytes written': 1000000000, 'HDFS: Number of read operations': 29, 'HDFS: Number of large read operations': 0, 'HDFS: Number of write operations': 2, 'HDFS: Number of bytes read erasure-coded': 0, 'Killed map tasks': 1, 'Launched map tasks': 9, 'Launched reduce tasks': 1, 'Rack-local map tasks': 9, 'Total time spent by all maps in occupied slots (ms)': 651540, 'Total time spent by all reduces in occupied slots (ms)': 431232, 'Total time spent by all map tasks (ms)': 162885, 'Total time spent by all reduce tasks (ms)': 53904, 'Total vcore-milliseconds taken by all map tasks': 162885, 'Total vcore-milliseconds taken by all reduce tasks': 53904, 'Total megabyte-milliseconds taken by all m

In [37]:
for file in log_files:
    num_workers_match = re.search(r"terasort_output_(\d+)_node", os.path.basename(file))
    if num_workers_match:
        num_workers = int(num_workers_match.group(1))
        print(f"Number of workers: {num_workers}")

         # Calculate aggregate resource utilization
        vcore_total = terasort_metrics.get('Total vcore-milliseconds taken by all map tasks', 0) + terasort_metrics.get('Total vcore-milliseconds taken by all reduce tasks', 0)
        agg_resource_util = vcore_total / num_workers
        print(f"Aggregate resource utilization: {agg_resource_util}")
        # Calculate shuffle throughput 

        shuffle_bytes = terasort_metrics.get('Reduce shuffle bytes', 0)
        reduce_time_ms = terasort_metrics.get('Total time spent by all reduce tasks (ms)', 0)
        shuffle_throughput = shuffle_bytes / (reduce_time_ms / 1000) / (1024 ** 2)
        print(f"Shuffle throughput: {shuffle_throughput} MB/s")
    else:
        num_workers = None


       



Number of workers: 6
Aggregate resource utilization: 36131.5
Shuffle throughput: 5.1414021294985295 MB/s
